In [60]:
# install required packages
%pip install langchain-ollama langchain-community rdflib

You should consider upgrading via the '/Users/fekaputr/Git/graphrag/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [61]:
from langchain_ollama import ChatOllama

# Create the LLM wrapper
ollama_llm = ChatOllama(
    base_url="https://ollama-gpt-oss.cluster.ai.wu.ac.at/",  # Ollama server URL
    model="gpt-oss:120b",                                    # name of the model you pulled
    temperature=0.5                                          # adjust temperature as needed              
)

In [62]:
from langchain.graphs import RdfGraph
from langchain.chains import GraphSparqlQAChain
from IPython.display import Markdown, display

# feeding the schema using a local RDF file
rdf_graph = RdfGraph(
    source_file="film.ttl",
    serialization="ttl",  # adjust if your file is xml/n3/etc.
    standard="owl",       # or "owl" if you’re using OWL
)
rdf_graph.load_schema()  # load the schema

# create the QA chain
chain = GraphSparqlQAChain.from_llm(
    allow_dangerous_requests=True,
    graph=rdf_graph,
    llm=ollama_llm,
    verbose=True,
)

In [63]:
# Query 1
result = chain.invoke({"query": "List all films and their properties."})
display(Markdown(result['result']))



> Entering new GraphSparqlQAChain chain...
Identified intent:
SELECT
Generated SPARQL:
PREFIX film: <http://semantics.id/ns/example/film#>
SELECT ?film ?budget ?duration ?releaseYear ?hasActor ?hasDirector ?hasGenre ?hasComposer ?hasCrew ?hasPerformer ?hasScriptWriter ?hasSoundtrack ?prequel ?sequel ?hasFilmStudio
WHERE {
  ?film a film:Film .
  OPTIONAL { ?film film:budget ?budget . }
  OPTIONAL { ?film film:duration ?duration . }
  OPTIONAL { ?film film:releaseYear ?releaseYear . }
  OPTIONAL { ?film film:hasActor ?hasActor . }
  OPTIONAL { ?film film:hasDirector ?hasDirector . }
  OPTIONAL { ?film film:hasGenre ?hasGenre . }
  OPTIONAL { ?film film:hasComposer ?hasComposer . }
  OPTIONAL { ?film film:hasCrew ?hasCrew . }
  OPTIONAL { ?film film:hasPerformer ?hasPerformer . }
  OPTIONAL { ?film film:hasScriptWriter ?hasScriptWriter . }
  OPTIONAL { ?film film:hasSoundtrack ?hasSoundtrack . }
  OPTIONAL { ?film film:prequel ?prequel . }
  OPTIONAL { ?film film:sequel ?sequel . }
  O

**Films and their known properties**

| Film | Year | Actors / Cast | Director | Genres | Writers | Production company | Other notable property |
|------|------|---------------|----------|--------|---------|--------------------|------------------------|
| **film_1** | 2018 | ewan_mcgregor, hayley_atwell | marc_forster | animation, family | writer_1, writer_2 | WaltDisneyPictures | – |
| **film_2** | 2014 | alicia_vikander, ewan_mcgregor | julius_avery | action, drama | julius_avery, writer_3 | EntertainmentOne | – |
| **film_3** | 2017 | dan_stevens, emma_watson, ewan_mcgregor | bill_condon | family, romance | writer_4, writer_5 | WaltDisneyPictures | character = Belle |
| **film_4** | 2021 | zendaya | (none listed) | science_fiction | (none listed) | (none listed) | composer = hans_zimmer |
| **film_5** | 1984 | kyle_maclachlan | (none listed) | science_fiction | (none listed) | (none listed) | – |

**Explanation of the columns**

* **Year** – the release year of the film.  
* **Actors / Cast** – individuals listed as actors for the film.  
* **Director** – the person credited as director.  
* **Genres** – the genre(s) assigned to the film (e.g., animation, family, action, drama, science_fiction, romance).  
* **Writers** – the writer(s) associated with the film.  
* **Production company** – the studio or company that produced the film.  
* **Other notable property** – any additional property that appeared in the data (for *film_3* a character “Belle” is recorded; for *film_4* a composer “hans_zimmer” is recorded).

All the information above is taken directly from the query results you provided.